# Baseline 2 — TabTransformer on Mendeley URL features
**Mục tiêu:** 12 URL features (no HTML) → TabTransformer → binary classification  
**Dataset:** `mendeley-phishing-2021` → `index.csv` (không cần giải nén html)  
**Thời gian:** ~20–40 phút trên GPU T4

In [ ]:
import os, sys, json, re, math, warnings
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
)
from tqdm.notebook import tqdm

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
warnings.filterwarnings('ignore')

SEED = 42; N_FOLDS = 5
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
PROJECT = Path('..')
RAW_DIR = PROJECT / 'data' / 'raw'
OUT_DIR = Path('/kaggle/working') if KAGGLE_INPUT.exists() else PROJECT / 'data'
MODEL_DIR = OUT_DIR / 'models'; FIG_DIR = OUT_DIR / 'figures'
MODEL_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect Mendeley index.csv anywhere under /kaggle/input/
mendeley_idx = RAW_DIR / 'mendeley' / 'index.csv'
if KAGGLE_INPUT.exists():
    for p in KAGGLE_INPUT.rglob('index.csv'):
        mendeley_idx = p; break
print(f'Mendeley idx: {mendeley_idx.exists()}')

In [ ]:
SUSPICIOUS_KEYWORDS = ['login','secure','verify','account','update','banking',
    'confirm','signin','password','reset','authenticate','paypal','webscr','free','bonus']
COMMON_TLDS = {'com','org','net','gov','edu','mil','io','co','uk',
               'au','de','jp','fr','ca','ru','cn','in','br','pl',
               'html','php','asp','jsp'}
URL_FEATURE_KEYS = ['url_length','domain_length','path_length','entropy',
    'special_char_ratio','digit_ratio','subdomain_count','has_https',
    'has_ip_address','suspicious_keywords','url_depth','tld_in_path']
TABULAR_DIM = 29

def shannon_entropy(text):
    if not text: return 0.0
    e, l = 0.0, len(text)
    for c in set(text):
        p = text.count(c) / l
        if p > 0: e -= p * math.log2(p)
    return round(e, 4)

def extract_url_features(url):
    parsed = urlparse(url)
    domain = (parsed.netloc or parsed.hostname or '').split(':')[0]
    path = parsed.path or ''
    fu = url.strip(); parts = domain.split('.')
    sc = sum(1 for c in fu if c in '@-_?.&=%+#~!'); dc = sum(1 for c in fu if c.isdigit())
    tc = max(len(fu), 1)
    ip_r = re.compile(r'^(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)$')
    return {
        'url_length': len(fu), 'domain_length': len(domain),
        'path_length': len(path), 'entropy': shannon_entropy(fu),
        'special_char_ratio': round(sc/tc, 4), 'digit_ratio': round(dc/tc, 4),
        'subdomain_count': max(0, len(parts)-2) if len(parts) >= 2 else 0,
        'has_https': 1 if parsed.scheme == 'https' else 0,
        'has_ip_address': 1 if ip_r.match(domain) else 0,
        'suspicious_keywords': sum(1 for kw in SUSPICIOUS_KEYWORDS if kw in fu.lower()),
        'url_depth': len([s for s in path.split('/') if s]),
        'tld_in_path': 1 if any(f'.{t}' in path.lower() for t in COMMON_TLDS) else 0,
    }

In [ ]:
df_full = pd.read_csv(mendeley_idx, encoding='utf-8')
print(f'Full dataset : {len(df_full):,} | Phishing: {(df_full["result"]==1).sum():,} | Genuine: {(df_full["result"]==0).sum():,}')

# Deterministic balanced sample — IDENTICAL to train_baseline2.py / evaluate.py
SAMPLE_SIZE = 50000
n_each = SAMPLE_SIZE // 2
df_p = df_full[df_full['result']==1].sample(n=min(n_each, (df_full['result']==1).sum()), random_state=SEED)
df_g = df_full[df_full['result']==0].sample(n=min(n_each, (df_full['result']==0).sum()), random_state=SEED)
df = pd.concat([df_p, df_g]).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'Sampled      : {len(df):,} | Phishing: {(df["result"]==1).sum():,} | Genuine: {(df["result"]==0).sum():,}')

# 12 URL features + 17 DNS/WHOIS/SSL defaults = 29-dim (RAW; scaler fitted per-fold later)
url_vectors = []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting URL features'):
    feats = extract_url_features(str(row['url']).strip())
    vec = [feats[k] for k in URL_FEATURE_KEYS]
    vec += [-1.0] * (TABULAR_DIM - len(vec))
    url_vectors.append(vec)

X = np.array(url_vectors, dtype=np.float32)
y = df['result'].values.astype(np.float32)
print(f'Shape: {X.shape}, Phishing: {y.sum()}, Benign: {(y==0).sum()}')

# Stratified 80/20 train/test split — test NEVER used in CV (same logic as evaluate.py)
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=SEED, stratify=df['result'].values
)
train_idx = np.array(train_idx, dtype=np.int64)
test_idx  = np.array(test_idx,  dtype=np.int64)
print(f'Train: {len(train_idx):,} | Test (held-out): {len(test_idx):,}')
with open(OUT_DIR / 'baseline2_splits.json', 'w') as f:
    json.dump({'train_indices': train_idx.tolist(), 'test_indices': test_idx.tolist()}, f, indent=2)
print(f'Saved: {OUT_DIR / "baseline2_splits.json"}')

# ── Dataset statistics (for thesis figures) ──
dataset_stats = {
    'dataset': 'Mendeley 2021',
    'n_full': int(len(df_full)),
    'n_full_phishing': int((df_full['result']==1).sum()),
    'n_full_benign': int((df_full['result']==0).sum()),
    'n_samples': int(len(df)),
    'n_phishing': int(y.sum()),
    'n_benign': int((y==0).sum()),
    'phishing_ratio': round(float(y.mean()), 6),
    'n_features': len(URL_FEATURE_KEYS),
    'feature_keys': URL_FEATURE_KEYS,
}
with open(OUT_DIR / 'dataset_stats_mendeley.json', 'w') as f:
    json.dump(dataset_stats, f, indent=2)
print(f'Stats saved to {OUT_DIR / "dataset_stats_mendeley.json"}')

In [ ]:
class FeatureEmbedding(nn.Module):
    def __init__(self, d=32): super().__init__(); self.e = nn.Linear(1, d)
    def forward(self, x): return self.e(x.unsqueeze(-1))

class TabTransformer(nn.Module):
    def __init__(self, nf=29, ed=32, nh=4, hd=256, od=128, dp=0.1):
        super().__init__()
        self.embs = nn.ModuleList([FeatureEmbedding(ed) for _ in range(nf)])
        self.attn = nn.MultiheadAttention(ed, nh, batch_first=True, dropout=dp)
        self.n1 = nn.LayerNorm(ed); self.n2 = nn.LayerNorm(ed)
        self.ff = nn.Sequential(nn.Linear(ed, hd), nn.GELU(), nn.Dropout(dp), nn.Linear(hd, ed), nn.Dropout(dp))
        self.proj = nn.Linear(ed * nf, od)
        self.cls = nn.Sequential(nn.Linear(od, 64), nn.ReLU(), nn.Dropout(dp), nn.Linear(64, 1))
    def forward(self, x):
        h = torch.stack([e(x[:, i]) for i, e in enumerate(self.embs)], 1)
        a, _ = self.attn(h, h, h); h = self.n1(h + a)
        f = self.ff(h); h = self.n2(h + f)
        return self.cls(self.proj(h.reshape(h.size(0), -1)))

In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X) if isinstance(X, np.ndarray) else X
        self.y = torch.from_numpy(y.reshape(-1,1).astype(np.float32)) if isinstance(y, np.ndarray) else y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def compute_metrics(labels, preds):
    pb = (preds >= 0.5).astype(int)
    fpr_val = 0.0
    if len(np.unique(labels)) > 1:
        cm  = confusion_matrix(labels, pb)
        tn, fp = cm[0, 0], cm[0, 1]
        fpr_val = round(fp / max(tn + fp, 1), 4)
    return {'accuracy': accuracy_score(labels, pb),
            'precision': precision_score(labels, pb, zero_division=0),
            'recall': recall_score(labels, pb, zero_division=0),
            'f1': f1_score(labels, pb, zero_division=0),
            'auc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else 0.0,
            'fpr': fpr_val}

def train_epoch(model, loader, opt, crit):
    model.train(); total = 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); loss = crit(model(Xb), yb)
        loss.backward(); opt.step(); total += loss.item() * Xb.size(0)
    return total / len(loader.dataset)

def evaluate(model, loader, crit):
    model.eval(); total = 0; preds, labs = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            logits = model(Xb)
            total += crit(logits, yb).item() * Xb.size(0)
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            labs.extend(yb.cpu().numpy())
    preds, labs = np.array(preds), np.array(labs)
    m = compute_metrics(labs, preds); m['loss'] = total / len(loader.dataset)
    return m, preds, labs

In [ ]:
BS, EP, LR = 128, 50, 1e-3
train_labels = y[train_idx]
pos_weight = torch.tensor([(len(train_labels) - train_labels.sum()) / max(train_labels.sum(), 1)], device=DEVICE)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f'pos_weight={pos_weight.item():.2f} (neg={int(len(train_labels)-train_labels.sum())}, pos={int(train_labels.sum())})')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_metrics, test_metrics = [], []
test_preds = []
folds_meta, history = [], []

for fold, (tr_rel, te_rel) in enumerate(skf.split(train_idx, train_labels)):
    tr_idx = train_idx[tr_rel]
    te_idx = train_idx[te_rel]
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    # Per-fold StandardScaler: fit on train fold ONLY, then transform test fold
    scaler = StandardScaler().fit(X[tr_idx])
    X_tr = scaler.transform(X[tr_idx]).astype(np.float32)
    X_te = scaler.transform(X[te_idx]).astype(np.float32)
    y_tr, y_te = y[tr_idx], y[te_idx]
    tr_ld = DataLoader(SimpleDataset(X_tr, y_tr), batch_size=BS, shuffle=True)
    te_ld = DataLoader(SimpleDataset(X_te, y_te), batch_size=BS)
    model = TabTransformer(nf=X_tr.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP)
    hist = []
    for ep in range(1, EP + 1):
        tl = train_epoch(model, tr_ld, opt, crit)
        m, _, _ = evaluate(model, te_ld, crit)
        sched.step()
        hist.append({'epoch': ep, 'train_loss': round(float(tl), 5),
                     'val_auc': round(float(m['auc']), 5), 'val_f1': round(float(m['f1']), 5)})
        if ep % 10 == 0:
            print(f'  Epoch {ep:2d}/{EP} | Loss: {tl:.4f} | AUC: {m["auc"]:.4f} | F1: {m["f1"]:.4f}')
    fm, _, _ = evaluate(model, te_ld, crit)
    fm['fold'] = fold + 1
    all_metrics.append(fm)
    history.append({'fold': fold + 1, 'epochs': hist})
    torch.save(model.state_dict(), MODEL_DIR / f'baseline2_fold{fold+1}.pt')
    folds_meta.append({
        'fold': int(fold + 1),
        'test_indices': te_idx.tolist(),
        'scaler_mean': scaler.mean_.tolist(),
        'scaler_scale': scaler.scale_.tolist(),
        'n_features': int(X_tr.shape[1]),
    })
    print(f'  Done (CV): Acc={fm["accuracy"]:.4f}, AUC={fm["auc"]:.4f}, F1={fm["f1"]:.4f}')

    # Held-out test evaluation using this fold's train scaler (test never in CV)
    X_ts = scaler.transform(X[test_idx]).astype(np.float32)
    test_ld = DataLoader(SimpleDataset(X_ts, y[test_idx]), batch_size=BS)
    tm, tp, _ = evaluate(model, test_ld, crit)
    tm['fold'] = fold + 1
    test_metrics.append(tm)
    test_preds.append(tp)
    print(f'  Test    : Acc={tm["accuracy"]:.4f}, AUC={tm["auc"]:.4f}, F1={tm["f1"]:.4f}')

# ── Save artifacts for evaluation/figures ──
with open(MODEL_DIR / 'baseline2_folds.json', 'w') as f:
    json.dump({'n_folds': N_FOLDS, 'folds': folds_meta}, f, indent=2)
with open(OUT_DIR / 'training_logs_baseline2.json', 'w') as f:
    json.dump(history, f, indent=2)
test_labels_arr = y[test_idx].astype(np.float32)
test_preds_mean = np.mean(np.stack(test_preds), axis=0)
tp_obj = np.empty(N_FOLDS, dtype=object)
for f in range(N_FOLDS):
    tp_obj[f] = test_preds[f]
np.savez(OUT_DIR / 'predictions_baseline2.npz',
         test_preds_mean=test_preds_mean, test_labels=test_labels_arr, test_preds=tp_obj)
print(f'Artifacts saved: baseline2_folds.json, training_logs_baseline2.json, predictions_baseline2.npz')

avg = {k: np.mean([m[k] for m in all_metrics]) for k in ['accuracy','precision','recall','f1','auc','fpr']}
std = {k: np.std([m[k] for m in all_metrics]) for k in ['accuracy','precision','recall','f1','auc','fpr']}
test_avg = {k: np.mean([m[k] for m in test_metrics]) for k in ['accuracy','precision','recall','f1','auc']}
test_std = {k: np.std([m[k] for m in test_metrics]) for k in ['accuracy','precision','recall','f1','auc']}
print(f'\n>>> {N_FOLDS}-Fold CV (train portion): Acc={avg["accuracy"]:.4f}+-{std["accuracy"]:.4f}, AUC={avg["auc"]:.4f}+-{std["auc"]:.4f}, F1={avg["f1"]:.4f}+-{std["f1"]:.4f}')
print(f'>>> Held-out test (never in CV): Acc={test_avg["accuracy"]:.4f}+-{test_std["accuracy"]:.4f}, AUC={test_avg["auc"]:.4f}+-{test_std["auc"]:.4f}, F1={test_avg["f1"]:.4f}+-{test_std["f1"]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']
a = axes.flatten()

cm = confusion_matrix(test_labels_arr, (test_preds_mean >= 0.5).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=a[0],
            xticklabels=['Benign','Phishing'], yticklabels=['Benign','Phishing'])
a[0].set_title('Baseline 2 — Confusion Matrix (held-out test)'); a[0].set_ylabel('True'); a[0].set_xlabel('Predicted')

for f in range(N_FOLDS):
    fpr, tpr, _ = roc_curve(test_labels_arr, test_preds[f])
    a[1].plot(fpr, tpr, color=colors[f], lw=1.5, alpha=0.7,
              label=f'Fold {f+1} (AUC={roc_auc_score(test_labels_arr, test_preds[f]):.4f})')
fpr, tpr, _ = roc_curve(test_labels_arr, test_preds_mean)
a[1].plot(fpr, tpr, 'k--', lw=2.5, label=f'Ensemble (AUC={roc_auc_score(test_labels_arr, test_preds_mean):.4f})')
a[1].plot([0,1],[0,1], 'gray', lw=1, alpha=0.5)
a[1].set_title('ROC Curves (held-out test)'); a[1].set_xlabel('FPR'); a[1].set_ylabel('TPR')
a[1].legend(fontsize=8, loc='lower right')

names = ['accuracy','precision','recall','f1','auc']
x = np.arange(len(names)); means = [test_avg[m] for m in names]; stdevs = [test_std[m] for m in names]
a[2].bar(x, means, yerr=stdevs, capsize=5, color='#e41a1c', alpha=0.8)
a[2].set_xticks(x); a[2].set_xticklabels([m.capitalize() for m in names])
a[2].set_ylim(0, 1); a[2].set_title('Metrics — Held-out Test (Mean+-Std)')
for i, (m, s) in enumerate(zip(means, stdevs)):
    a[2].text(i, m + s + 0.02, f'{m:.3f}+-{s:.3f}', ha='center', fontsize=8)

# Training curves — mean across folds
av = []
max_ep = max(len(h['epochs']) for h in history)
for ep in range(1, max_ep + 1):
    rows = [h['epochs'][ep-1] for h in history if len(h['epochs']) >= ep]
    av.append({'epoch': ep, **{k: float(np.mean([r[k] for r in rows])) for k in ['train_loss','val_auc','val_f1']}})
a[3].plot([r['epoch'] for r in av], [r['train_loss'] for r in av], 'o-', color='#d62728', lw=1.5, label='Train loss')
a[3].set_xlabel('Epoch'); a[3].set_ylabel('Loss', color='#d62728')
a[3].tick_params(axis='y', labelcolor='#d62728')
a3b = a[3].twinx()
a3b.plot([r['epoch'] for r in av], [r['val_auc'] for r in av], 's-', color='#1f77b4', lw=1.5, label='Val AUC')
a3b.plot([r['epoch'] for r in av], [r['val_f1'] for r in av], 'd-', color='#2ca02c', lw=1.5, label='Val F1')
a3b.set_ylim(0, 1); a3b.set_ylabel('Score')
a3b.legend(fontsize=8, loc='lower left')
a[3].set_title('Training Curves (mean across folds)')

# Data distribution (full + sampled)
w = 0.38; xs = np.arange(2)
a[4].bar(xs - w/2, [dataset_stats['n_full_benign'], dataset_stats['n_full_phishing']], w,
         label='Full', color=['#2ca02c','#d62728'], alpha=0.45)
a[4].bar(xs + w/2, [dataset_stats['n_benign'], dataset_stats['n_phishing']], w,
         label='Sampled', color=['#2ca02c','#d62728'], alpha=0.9)
a[4].set_xticks(xs); a[4].set_xticklabels(['Benign','Phishing'])
a[4].set_ylabel('Samples')
a[4].set_title(f"Mendeley Distribution (full={dataset_stats['n_full']:,}, sampled={dataset_stats['n_samples']:,})")
a[4].legend(fontsize=8)

a[5].axis('off')
a[5].text(0.02, 0.95, 'Baseline 2 — TabTransformer (Mendeley URL)', fontsize=13, weight='bold', va='top')
a[5].text(0.02, 0.75, f"Held-out test (5-fold ensemble):\nAcc={test_avg['accuracy']:.4f}+-{test_std['accuracy']:.4f}\n"
                      f"AUC={test_avg['auc']:.4f}+-{test_std['auc']:.4f}\n"
                      f"F1={test_avg['f1']:.4f}+-{test_std['f1']:.4f}\n"
                      f"Precision={test_avg['precision']:.4f}+-{test_std['precision']:.4f}\n"
                      f"Recall={test_avg['recall']:.4f}+-{test_std['recall']:.4f}", fontsize=11, va='top')

plt.tight_layout(); plt.savefig(FIG_DIR / 'baseline2_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "baseline2_summary.png"}')

In [ ]:
results = {
    'model': 'Baseline 2 - TabTransformer (Mendeley URL)',
    'n_folds': N_FOLDS,
    'sample_size': len(df),
    **{k: round(float(test_avg[k]), 6) for k in test_avg},
    **{k + '_std': round(float(test_std[k]), 6) for k in test_std},
    **{'cv_' + k: round(float(avg[k]), 6) for k in avg},
    **{'cv_' + k + '_std': round(float(std[k]), 6) for k in std},
}
with open(MODEL_DIR / 'evaluation_baseline2.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {MODEL_DIR / "evaluation_baseline2.json"}')

---
### Download từ Output tab:
- `figures/baseline2_summary.png` (báo cáo)
- `data/models/evaluation_baseline2.json`
- `data/models/baseline2_folds.json`
- `baseline2_splits.json`
- `training_logs_baseline2.json`
- `predictions_baseline2.npz`
- `dataset_stats_mendeley.json`
- `data/models/baseline2_fold1..5.pt` (optional)

Sau đó chạy **Proposed Model** → `kaggle_proposed.ipynb`